# ECOSTRESS LST Quality Control (QC) Masking

This notebook applies quality control (QC) flags to ECOSTRESS Land Surface Temperature (LST) GeoTIFF files downloaded from NASA AppEEARS.

## Overview
1. **Inventory** — Pairs each LST `.tif` file with its corresponding QC `.tif` file by matching acquisition datetime and tile ID parsed from filenames.
2. **Organize** — Copies raw LST and QC files into month-sorted subfolders for easier navigation.
3. **QC Masking** — For each paired scene:
   - Converts LST values from raw counts to degrees Celsius
   - Reads the QC band and masks any pixels with QC flag ≠ 0 (perfect) or 1 (nominal quality)
   - Skips scenes where >90% of pixels are masked (low quality)
   - Saves the cleaned LST raster as a `float32` GeoTIFF
4. **Inspect** — Plots an example QC'd scene to verify output.

## Expected Input File Structure
Files should follow the standard ECOSTRESS L2T filename convention:
```
ECO_L2T_LSTE.002_LST_doy<YYYYDDDHHMMSS>_aid0001_<tileID>.tif
ECO_L2T_LSTE.002_QC_doy<YYYYDDDHHMMSS>_aid0001_<tileID>.tif
```

## QC Flag Reference (bits 0–1)
| Value | Meaning |
|-------|---------|
| 0 | Good quality |
| 1 | Nominal quality |
| 2 | Cloud |
| 3 | Not produced |

Only pixels with QC = 0 or 1 are retained.

## 1. Setup

In [ ]:
# Mount Google Drive (required when running in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install rioxarray if not already available
!pip install rioxarray

In [ ]:
import os
from os import makedirs
from os.path import join, basename, splitext
from glob import glob
from datetime import datetime
import re
import shutil

import numpy as np
import pandas as pd
import rasterio
import rioxarray
import matplotlib.pyplot as plt

## 2. Configuration

**Update the paths below** to point to your own Google Drive folders:
- `input_directory`: folder containing raw ECOSTRESS LST and QC `.tif` files
- `output_directory`: folder where QC-masked LST files will be saved

In [ ]:
# --- USER CONFIGURATION ---
# Update these paths to match your Google Drive folder structure
input_directory  = "/content/drive/MyDrive/YOUR_PROJECT/ECOSTRESS/LST"       # folder with raw LST + QC .tif files
output_directory = "/content/drive/MyDrive/YOUR_PROJECT/ECOSTRESS/QCd_LST"  # folder for QC-masked output files
# --------------------------

# Create output directory if it does not already exist
makedirs(output_directory, exist_ok=True)

# Display settings
pd.set_option("display.max_colwidth", None)  # show full file paths in dataframe previews

# Plot aesthetics
alpha         = 0.7    # transparency for overlay plots
fig_width_px  = 1080   # figure width in pixels
fig_height_px = 720    # figure height in pixels

## 3. Build File Inventory

Scan the input directory for LST and QC files. Parse each filename to extract:
- **Acquisition datetime** (UTC) from the `doyYYYYDDDHHMMSS` component
- **Tile ID** (e.g., `15N`, `16N`) from the end of the filename

Then merge the two lists on `(datetime_UTC, tile_id)` to produce matched LST–QC pairs.

In [ ]:
def extract_datetime_from_filename(filename):
    """Parse acquisition datetime (UTC) from ECOSTRESS L2T filename.

    Expected pattern: doy<YYYY><DDD><HH><MM><SS>
    Example: ECO_L2T_LSTE.002_LST_doy2023127025906_aid0001_16N.tif
             -> 2023-05-07 02:59:06 UTC
    """
    base = basename(filename)
    match = re.search(r'doy(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})', base)
    if match:
        year, doy, hour, minute, second = match.groups()
        return datetime.strptime(f"{year}{doy}{hour}{minute}{second}", "%Y%j%H%M%S")
    else:
        raise ValueError(f"Could not extract datetime from filename: {filename}")


def extract_tile_id(filename):
    """Parse the MGRS tile ID (e.g., '15N', '16N') from an ECOSTRESS L2T filename.

    Expected pattern: ..._<##N|S>.tif at end of filename.
    """
    match = re.search(r'_(\d{2}[NS])\.tif$', filename)
    if match:
        return match.group(1)
    else:
        raise ValueError(f"Could not extract tile ID from filename: {filename}")


# --- Inventory LST files ---
ST_raw_filenames = pd.DataFrame({
    "ST_raw_filename": sorted(glob(join(input_directory, "*_LST_*.tif")))
})
ST_raw_filenames["datetime_UTC"] = ST_raw_filenames["ST_raw_filename"].apply(extract_datetime_from_filename)
ST_raw_filenames["tile_id"]      = ST_raw_filenames["ST_raw_filename"].apply(extract_tile_id)

# --- Inventory QC files ---
QC_filenames = pd.DataFrame({
    "QC_filename": sorted(glob(join(input_directory, "*_QC_*.tif")))
})
QC_filenames["datetime_UTC"] = QC_filenames["QC_filename"].apply(extract_datetime_from_filename)
QC_filenames["tile_id"]      = QC_filenames["QC_filename"].apply(extract_tile_id)

# --- Merge on (datetime_UTC, tile_id) to pair each LST scene with its QC file ---
ST_filenames = pd.merge(ST_raw_filenames, QC_filenames, on=["datetime_UTC", "tile_id"])

# --- Construct output paths (preserve original filename, write to output_directory) ---
ST_filenames["ST_masked_filename"] = ST_filenames["ST_raw_filename"].apply(
    lambda fname: join(output_directory, basename(fname))
)

# --- Reorder columns for readability ---
ST_filenames = ST_filenames[[
    "datetime_UTC", "tile_id", "ST_raw_filename", "QC_filename", "ST_masked_filename"
]]

print(f"Found {len(ST_filenames)} matched LST–QC scene pairs.")
ST_filenames

## 4. Organize Files by Month

Copy raw LST and QC files into `YYYY-MM` subfolders within the output directory.
This step is optional but helps keep large time series organised.

In [ ]:
# Base directories for monthly subfolders
LST_monthly_dir = join(output_directory, "LST_by_month")
QC_monthly_dir  = join(output_directory, "QC_by_month")

os.makedirs(LST_monthly_dir, exist_ok=True)
os.makedirs(QC_monthly_dir,  exist_ok=True)


def get_month_str(dt):
    """Return a 'YYYY-MM' string for a given datetime object."""
    return dt.strftime("%Y-%m")


# Copy files into month-sorted subfolders
for idx, row in ST_filenames.iterrows():
    month_str = get_month_str(row["datetime_UTC"])

    lst_month_dir = join(LST_monthly_dir, month_str)
    qc_month_dir  = join(QC_monthly_dir,  month_str)
    os.makedirs(lst_month_dir, exist_ok=True)
    os.makedirs(qc_month_dir,  exist_ok=True)

    shutil.copy2(row["ST_raw_filename"], join(lst_month_dir, basename(row["ST_raw_filename"])))
    shutil.copy2(row["QC_filename"],     join(qc_month_dir,  basename(row["QC_filename"])))

print("Files copied successfully into monthly subfolders.")

## 5. Apply QC Masking and Save

For each matched LST–QC pair:
1. Open the LST raster and convert pixel values from raw integer to **degrees Celsius**.
   - Values outside the physically plausible Kelvin range (100–400 K) are set to NaN.
   - Then subtract 273.15 to convert K → °C.
2. Read the QC band and isolate the two least-significant bits (overall quality flag).
3. Mask pixels with QC flag values of 2 (cloud) or 3 (not produced).
4. Skip scenes where more than 90 % of pixels are masked (too little usable data).
5. Write the masked `float32` GeoTIFF to `output_directory`.

In [ ]:
def preprocess_QC_file(file_QC):
    """Read a QC GeoTIFF and return a 2-D array of the overall quality flag.

    The ECOSTRESS L2T QC layer encodes overall quality in bits 0–1:
      0 = good quality
      1 = nominal quality
      2 = cloud contaminated
      3 = not produced

    Parameters
    ----------
    file_QC : str
        Path to the QC GeoTIFF.

    Returns
    -------
    numpy.ndarray
        2-D integer array with values 0–3.
    """
    with rasterio.open(file_QC, 'r') as f_QC:
        QC_img = f_QC.read(1).astype(int)   # read first band as signed int
        QC_img[QC_img == -99999] = -1        # replace ECOSTRESS nodata sentinel
        QC_img_2 = QC_img & 0b11             # isolate bits 0–1 (overall quality)
    return QC_img_2


# Threshold: skip scenes with more than this fraction of masked pixels
MISSING_THRESHOLD = 0.90

saved_count  = 0
skipped_count = 0

for i, (datetime_UTC, tile_id, ST_raw_filename, QC_filename, ST_masked_filename) in ST_filenames.iterrows():

    # --- Load LST raster ---
    ST = rioxarray.open_rasterio(ST_raw_filename).squeeze("band", drop=True)

    # Convert to float32 and apply physical range mask (100–400 K covers all realistic LST)
    ST.data = ST.data.astype("float32")
    ST.data[(ST.data < 100) | (ST.data > 400)] = np.nan

    # Convert from Kelvin to Celsius
    ST.data = ST.data - 273.15

    # --- Apply QC mask ---
    QC_img_2 = preprocess_QC_file(QC_filename)
    # Keep only QC = 0 (good) and QC = 1 (nominal); mask everything else
    ST.data = np.where((QC_img_2 != 0) & (QC_img_2 != 1), np.nan, ST.data)

    # --- Check data coverage ---
    missing_proportion = np.count_nonzero(np.isnan(ST.data)) / ST.data.size
    print(f"[{datetime_UTC}  tile={tile_id}]  missing: {missing_proportion * 100:.1f}%")

    if missing_proportion > MISSING_THRESHOLD:
        print("  -> Low quality scene. Skipping.")
        skipped_count += 1
        continue

    # --- Save masked output ---
    with rasterio.open(ST_raw_filename) as src:
        out_meta = src.meta.copy()

    out_meta.update({
        "driver": "GTiff",
        "height": ST.shape[0],
        "width":  ST.shape[1],
        "dtype":  "float32"
    })

    with rasterio.open(ST_masked_filename, 'w', **out_meta) as output_raster:
        output_raster.write(ST.data.astype('float32'), 1)

    saved_count += 1
    print("  -> Saved.")

print(f"\nProcessing complete.")
print(f"  Scenes saved:   {saved_count}")
print(f"  Scenes skipped: {skipped_count}")

## 6. Visual Inspection

Plot the first successfully saved QC-masked LST scene to verify that the masking
and unit conversion look correct.

In [ ]:
# Identify saved output files (those that actually exist on disk)
saved_files = [
    f for f in ST_filenames["ST_masked_filename"]
    if os.path.exists(f)
]

if not saved_files:
    print("No saved output files found. Run the QC masking step first.")
else:
    example_file = saved_files[0]  # use the first available scene

    with rasterio.open(example_file) as src:
        lst_data = src.read(1).astype(float)
        if src.nodata is not None:
            lst_data[lst_data == src.nodata] = np.nan

    plt.figure(figsize=(10, 6))
    plt.imshow(lst_data, cmap='inferno')
    plt.colorbar(label='LST (°C)')
    plt.title(f"QC-masked Land Surface Temperature\n{basename(example_file)}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()